# Demo — comparing four ways to demultiplex the same pool

The same pooled scRNA-seq sample (`Pool_ctr`, 10,308 cells) was demultiplexed four times:

| Run | Vireo mode | Cell lines given |
|---|---|---|
| **Guided, 7 lines** | reference-guided | RKO, HCT116, HT29, HCT8, CACO2, DLD1, LIM1215 |
| **Unguided** | genotype-free, 7 donors, then matched to the database | none |
| **Guided, no HCT8** | reference-guided | RKO, HCT116, HT29, CACO2, DLD1, LIM1215 |
| **Guided, no DLD1** | reference-guided | RKO, HCT116, HT29, HCT8, CACO2, LIM1215 |

**DLD1 and HCT8 come from the same donor**, so their genotypes are nearly identical.
Watch what each run does with those cells.

Run each cell from top to bottom. To compare your own group's run, change its entry in `RUNS` below.

In [1]:
DEMO_DIR = '/groups/nils/data/scrna_pipeline/demo/results/demux'

# label shown in the plots -> demux run id (folder under DEMO_DIR)
# Replace a precomputed_* id with your group's id once your run has finished.
RUNS = {
    'Guided, 7 lines': 'precomputed_guided7',
    'Unguided':        'precomputed_unguided',
    'Guided, no HCT8': 'precomputed_no_hct8',
    'Guided, no DLD1': 'precomputed_no_dld1',
}
REFERENCE_RUN = 'Guided, 7 lines'   # the run the others are compared against

In [2]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

LINES = ['RKO', 'HCT116', 'HT29', 'CACO2', 'LIM1215', 'DLD1', 'HCT8']
SIBLING_DOUBLET = 'HCT8+DLD1 doublet'
CATEGORIES = LINES + [SIBLING_DOUBLET, 'Other doublet', 'Unmatched donor', 'Unassigned']

# One fixed colour per category in every chart (validated categorical order; greys for non-lines)
COLORS = {
    'RKO': '#2a78d6', 'HCT116': '#eb6834', 'HT29': '#1baf7a', 'CACO2': '#eda100',
    'LIM1215': '#e87ba4', 'DLD1': '#008300', 'HCT8': '#6250d6',
    SIBLING_DOUBLET: '#3f3e3b', 'Other doublet': '#8a8983',
    'Unmatched donor': '#b5b4ad', 'Unassigned': '#dcdbd5',
}
FONT = dict(family='Arial, sans-serif', size=13, color='#0b0b0b')


def categorise(label):
    if label == 'unassigned':
        return 'Unassigned'
    if label.startswith('doublet:'):
        parts = set(label[len('doublet:'):].split('+'))
        return SIBLING_DOUBLET if parts == {'HCT8', 'DLD1'} else 'Other doublet'
    if label.startswith('vireo:'):
        return 'Unmatched donor'
    return label


labels = {}
for name, run_id in RUNS.items():
    path = os.path.join(DEMO_DIR, run_id, 'final_assignments.tsv')
    if not os.path.exists(path):
        print(f'{name}: {path} not found yet — skipped')
        continue
    fa = pd.read_csv(path, sep='\t', index_col='barcode')
    labels[name] = fa['cell_line'].map(categorise)

cats = pd.DataFrame(labels)
print(f'Loaded {len(cats.columns)} runs, {len(cats):,} cells each')

Loaded 4 runs, 10,308 cells each


## 1. Headline numbers

How many cells did each run confidently assign to a single cell line?

In [3]:
summary = pd.DataFrame({
    'assigned to a line': cats.isin(LINES).sum(),
    '% assigned': (100 * cats.isin(LINES).mean()).round(1),
    'HCT8+DLD1 doublet': (cats == SIBLING_DOUBLET).sum(),
    'other doublet': (cats == 'Other doublet').sum(),
    'unmatched donor': (cats == 'Unmatched donor').sum(),
    'unassigned': (cats == 'Unassigned').sum(),
})
summary

,assigned to a line,% assigned,HCT8+DLD1 doublet,other doublet,unmatched donor,unassigned
"Guided, 7 lines",8338,80.9,198,503,0,1269
Unguided,6508,63.1,0,1335,1803,662
"Guided, no HCT8",8649,83.9,0,540,0,1119
"Guided, no DLD1",8636,83.8,0,549,0,1123


## 2. Cells per label, per run

Each cell of the grid is the number of cells a run gave that label. Read across a row to see how one
label changes between runs; read down a column to see one run's composition.

In [4]:
counts = cats.apply(lambda s: s.value_counts()).reindex(CATEGORIES).fillna(0).astype(int)

BLUES = [[0.0, '#f7fafe'], [0.02, '#cde2fb'], [0.15, '#86b6ef'], [0.4, '#3987e5'],
         [0.7, '#1c5cab'], [1.0, '#0d366b']]

def heatmap_annotations(z, x, y, fmt, xref='x', yref='y', dark_above=None):
    vmax = z.max().max() if dark_above is None else dark_above
    notes = []
    for i, row in enumerate(y):
        for j, col in enumerate(x):
            v = z.iloc[i, j]
            if v == 0:
                continue
            notes.append(dict(x=col, y=row, text=fmt(v), showarrow=False, xref=xref, yref=yref,
                              font=dict(size=12, color='white' if v > 0.45 * vmax else '#0b0b0b')))
    return notes

fig = go.Figure(go.Heatmap(
    z=counts.values, x=counts.columns, y=counts.index, colorscale=BLUES, showscale=False,
    xgap=2, ygap=2, hovertemplate='%{y}<br>%{x}: %{z:,} cells<extra></extra>'))
fig.update_layout(
    annotations=heatmap_annotations(counts, counts.columns, counts.index, lambda v: f'{v:,}'),
    yaxis=dict(autorange='reversed'), xaxis=dict(side='top'),
    height=520, width=760, margin=dict(l=150, r=20, t=60, b=20),
    plot_bgcolor='white', paper_bgcolor='white', font=FONT)
fig.show()

## 3. Cell by cell: where do the reference run's cells go?

Rows are the labels from the reference run, columns the labels from the other run, for **the same cells**.
Each row adds up to 100 %. A perfect agreement would be a diagonal.

In [5]:
others = [c for c in cats.columns if c != REFERENCE_RUN]
fig = make_subplots(rows=1, cols=len(others), subplot_titles=others, horizontal_spacing=0.07,
                    shared_yaxes=True)
notes = []
for k, other in enumerate(others, start=1):
    ct = pd.crosstab(cats[REFERENCE_RUN], cats[other])
    rows = [c for c in CATEGORIES if c in ct.index]
    cols = [c for c in CATEGORIES if c in ct.columns]
    ct = ct.reindex(index=rows, columns=cols, fill_value=0)
    pct = ct.div(ct.sum(axis=1), axis=0) * 100
    fig.add_trace(go.Heatmap(
        z=pct.values, x=cols, y=rows, zmin=0, zmax=100, colorscale=BLUES, showscale=False,
        xgap=2, ygap=2, customdata=ct.values,
        hovertemplate=f'{REFERENCE_RUN}: %{{y}}<br>{other}: %{{x}}<br>%{{z:.0f}} % of row (%{{customdata:,}} cells)<extra></extra>'),
        row=1, col=k)
    xr, yr = ('x', 'y') if k == 1 else (f'x{k}', f'y{k}')
    notes += heatmap_annotations(pct, cols, rows, lambda v: f'{v:.0f}' if v >= 1 else '',
                                 xref=xr, yref=yr, dark_above=100)
fig.update_yaxes(autorange='reversed')
fig.update_xaxes(tickangle=45)
fig.update_layout(annotations=list(fig.layout.annotations) + notes,
                  height=560, width=480 * len(others), margin=dict(l=150, r=20, t=50, b=150),
                  plot_bgcolor='white', paper_bgcolor='white', font=FONT)
fig.update_yaxes(title_text=f'{REFERENCE_RUN} label', row=1, col=1)
fig.show()

## 4. The DLD1 / HCT8 family

DLD1 and HCT8 share a donor. Below are all cells that **any** run called DLD1, HCT8 or an HCT8+DLD1 doublet,
and how each run labelled them.

In [6]:
family_labels = ['DLD1', 'HCT8', SIBLING_DOUBLET]
family = cats[cats.isin(family_labels).any(axis=1)]
print(f'{len(family):,} cells were placed in the DLD1/HCT8 family by at least one run')

shown = family_labels + ['Other doublet', 'Unassigned', 'Unmatched donor']
fam_counts = family.apply(lambda s: s.where(s.isin(shown), 'Other line').value_counts())
order = shown + ['Other line']
fam_counts = fam_counts.reindex(order).fillna(0).astype(int)
fig = go.Figure()
for label in order:
    if fam_counts.loc[label].sum() == 0:
        continue
    other_line = label == 'Other line'
    fig.add_trace(go.Bar(
        y=fam_counts.columns, x=fam_counts.loc[label], name=label, orientation='h',
        marker=dict(color='white' if other_line else COLORS[label],
                    pattern=dict(shape='/', fgcolor='#8a8983', solidity=0.3) if other_line else None,
                    line=dict(color='#8a8983' if other_line else 'white', width=1 if other_line else 2)),
        hovertemplate=f'{label}<br>%{{y}}: %{{x:,}} cells<extra></extra>'))
fig.update_layout(barmode='stack', height=380, width=820,
                  xaxis=dict(title='cells', gridcolor='#ecebe7'), yaxis=dict(autorange='reversed'),
                  legend=dict(orientation='h', y=1.02, yanchor='bottom', x=0, traceorder='normal'),
                  margin=dict(l=150, r=20, t=70, b=50),
                  plot_bgcolor='white', paper_bgcolor='white', font=FONT)
fig.show()

347 cells were placed in the DLD1/HCT8 family by at least one run


## 5. Unguided run: which donor matched which cell line?

The unguided run first clusters cells into 7 anonymous donors, then compares each donor's genotype with every
reference in the database. `concordance` is the fraction of shared SNP positions where the two agree;
`gap` is how far the best match beats the runner-up. Donors without a confident match stay `no_match`.

In [7]:
unguided = [n for n in RUNS if n.lower().startswith('unguided')]
for name in unguided:
    path = os.path.join(DEMO_DIR, RUNS[name], 'donor_matches.tsv')
    if os.path.exists(path):
        dm = pd.read_csv(path, sep='\t')
        display(dm[['vireo_donor', 'cell_line', 'concordance', 'second_concordance', 'gap', 'confidence']]
                .fillna({'cell_line': '—'}).round(3))
    else:
        print(f'{name}: donor_matches.tsv not found yet')

,vireo_donor,cell_line,concordance,second_concordance,gap,confidence
0,donor0,—,0.709,0.834,-0.125,no_match
1,donor1,—,0.746,0.789,-0.044,no_match
2,donor2,RKO,0.896,0.700,0.196,high
3,donor3,HT29,0.825,0.744,0.081,low
4,donor4,—,0.741,0.743,-0.002,no_match
5,donor5,—,0.719,0.827,-0.108,no_match
6,donor6,HCT116,0.869,0.750,0.119,high


## Discussion

- Which run assigned the most cells to a single line? Is *more* always *better*?
- What happens to HCT8 cells when HCT8 is left out of the guided run — and to DLD1 cells when DLD1 is left out?
- In the 7-line guided run, why are so many cells called an "HCT8+DLD1 doublet"?
- The unguided run needs no prior knowledge. What does it get right, and where does it struggle?